In [ ]:
import json
from eval.utils import tranform_str_to_json
from eval.eval_molund import check_string_type
from eval.eval_molund import eval_molund_from_list
print("success")

success


In [4]:
def evaluate_molund_score(model_name):
    task_dict = dict(
        fg_samples="fg_samples", murcko='murcko_scaffold', ring_count='ring_count',
        ring_system='ring_system_scaffold', mutated='mutated', permutated='permutated'
    )
    pred_key_dict = dict(
        fg_samples="count", murcko='Output Scaffold', ring_count='count',
        ring_system='output', mutated='output', permutated='output'
    )
    gt_key_dict = dict(
        fg_samples="fg_num", murcko='largest_scaffold', ring_count='count',
        ring_system='', mutated='', permutated=''
    )
    result_dict = dict()
    
    for task in task_dict.keys():
        print(model_name, task)
        if 'llama' not in model_name:
            file_name = f"logs/{task_dict[task]}/{model_name}.json"
            
        pred_results = json.load(open(file_name, "r"))
        invalid_number = 0
        
        pred_list, gt_list = list(), list()
        for pred in pred_results:  
            if type(pred['json_results']) is str:
                pred_json = tranform_str_to_json(str_input=pred['json_results'])
                # if model_name == 'gemini': pred_json = pred_json[0]
                if pred_json == None:
                    invalid_number += 1
                    continue
                else:
                    if pred_key_dict[task] not in pred_json.keys():
                        invalid_number += 1; continue
                    if pred_json[pred_key_dict[task]] == "": 
                        invalid_number += 1; continue
                    if task in ["ring_count", "fg_samples"]:
                        if check_string_type(pred_json[pred_key_dict[task]]) == "string":
                            invalid_number += 1; continue;
                    pred_list.append(pred_json[pred_key_dict[task]])
                    if gt_key_dict[task] != "":
                        gt_list.append(pred[gt_key_dict[task]])
            else:
                if pred_key_dict[task] not in pred['json_results'].keys():
                    invalid_number += 1; continue
                if pred['json_results'][pred_key_dict[task]] == "": 
                    invalid_number += 1; continue
                if task in ["ring_count", "fg_samples"]:
                    if check_string_type(pred['json_results'][pred_key_dict[task]]) == "string":
                            invalid_number += 1; continue
                pred_list.append(pred['json_results'][pred_key_dict[task]])
                if gt_key_dict[task] != "":
                    gt_list.append(pred[gt_key_dict[task]])
        
        assert len(pred_results) == invalid_number+len(pred_list)
        result_dict[task] = eval_molund_from_list(gt_list=gt_list, pred_list=pred_list, total_number=len(pred_results), task=task)
        print(model_name, task, result_dict[task])
    
    print(f"eval_score_{model_name}", result_dict)
    # json.dump(result_dict, open(f"logs/eval_score_{model_name}.json", "w"), indent=4)


if __name__ == "__main__":
    model_list = ['qwen3-8b']
    for model_name in model_list:
        evaluate_molund_score(model_name=model_name)

qwen3-8b fg_samples
qwen3-8b fg_samples {'score': 0.49, 'fg_samples-valid-rate': 1.0}
qwen3-8b murcko
qwen3-8b murcko {'score': 0.06569255468905248, 'murcko-valid-rate': 0.75}
qwen3-8b ring_count
qwen3-8b ring_count {'score': 0.7, 'ring_count-valid-rate': 1.0}
qwen3-8b ring_system
qwen3-8b ring_system {'score': 0.8, 'ring_system-valid-rate': 1.0}
qwen3-8b mutated
qwen3-8b mutated {'score': 0.8, 'mutated-valid-rate': 1.0}
qwen3-8b permutated
qwen3-8b permutated {'score': 0.54, 'permutated-valid-rate': 1.0}
eval_score_qwen3-8b {'fg_samples': {'score': 0.49, 'fg_samples-valid-rate': 1.0}, 'murcko': {'score': 0.06569255468905248, 'murcko-valid-rate': 0.75}, 'ring_count': {'score': 0.7, 'ring_count-valid-rate': 1.0}, 'ring_system': {'score': 0.8, 'ring_system-valid-rate': 1.0}, 'mutated': {'score': 0.8, 'mutated-valid-rate': 1.0}, 'permutated': {'score': 0.54, 'permutated-valid-rate': 1.0}}
